In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
import cstarpy.integration
import os

COMPOUND_FILE = "cd8_limma_merged_filtered_targets_ic50_dpd.csv"
SELECTION     = "top4_bottom4_selected_modules_drugs_ic50.csv"   # from prep notebook
out_dir       = "02_outputs"
os.makedirs(out_dir, exist_ok=True)

# Load the collapsed-module selection built in prep (module column already exists)
drug_gene = pd.read_csv(SELECTION).rename(columns={"compound_name": "drug"})

modules  = drug_gene["module"].drop_duplicates().tolist()
exp_list = drug_gene["drug"].drop_duplicates().tolist()

print(f"Modules ({len(modules)}): {modules}")
print(f"Experiments ({len(exp_list)}): {exp_list}")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Modules (7): ['ARFGAP', 'BCL', 'IGF1R', 'JAK', 'MTOR', 'NAE', 'SERCA']
Experiments (12): ['QS-11', 'navitoclax', 'BMS-536924', 'AT9283', 'TG-101348', 'CYT-387', 'ruxolitinib', 'Deforolimus', 'Temsirolimus', 'Sapanisertib', 'Pevonedistat', 'Thapsigargin']


In [2]:
df = pd.read_csv(COMPOUND_FILE)

df_avg = (
    df.groupby(["compound_name", "gene"])
    .agg(logFC=("logFC", "mean"), adj_P_Val=("adj.P.Val", "mean"))
    .reset_index()
)
df_avg["logFC_thresh"] = np.where(df_avg["adj_P_Val"] < 0.05, df_avg["logFC"], 0)

x_df = df_avg.pivot_table(
    index="gene", columns="compound_name", values="logFC_thresh", aggfunc="first"
).fillna(0)[exp_list]

genes = x_df.index.tolist()
x     = x_df.values
print(f"x (expression) shape: {x.shape}  (genes × experiments)")

x (expression) shape: (15045, 12)  (genes × experiments)


In [3]:
dose_info = df.groupby("compound_name")["dose_uM"].first()
mech_info = df.groupby("compound_name")["mechanism"].first()

inhib_conc_matrix_top4_bottom4 = np.zeros((len(modules), len(exp_list)))
ic50_matrix_top4_bottom4       = np.ones((len(modules), len(exp_list))) * np.inf   # inf → g=1
gamma_matrix_top4_bottom4      = np.zeros((len(modules), len(exp_list)))            # 0 = inhibitor, >0 = activator
valid_matrix_top4_bottom4      = np.zeros((len(modules), len(exp_list)), dtype=bool)  # has real dose+IC50 data

GAMMA = 1.0   # activation coefficient, applied only where mechanism == "Activator"

for _, row in drug_gene.iterrows():
    if row["module"] in modules and row["drug"] in exp_list:
        i = modules.index(row["module"])
        j = exp_list.index(row["drug"])
        dose = dose_info.get(row["drug"], np.nan)
        ic50_raw = df[df["compound_name"] == row["drug"]]["IC50_nM"]
        ic50_nm = ic50_raw.iloc[0] if len(ic50_raw) else np.nan
        ic50 = ic50_nm / 1000 if pd.notna(ic50_nm) else np.nan
        mech = mech_info.get(row["drug"], "Inhibitor")
        if pd.notna(dose) and pd.notna(ic50):
            inhib_conc_matrix_top4_bottom4[i, j] = dose
            ic50_matrix_top4_bottom4[i, j]       = ic50
            gamma_matrix_top4_bottom4[i, j]      = GAMMA if mech == "Activator" else 0.0
            valid_matrix_top4_bottom4[i, j]      = True
        else:
            print(f"WARNING: missing dose/IC50 for module={row['module']} drug={row['drug']} "
                  f"— excluded from fit (not treated as 'no effect')")

# y_true = (1 + gamma_matrix * dose/IC50) / (1 + dose/IC50)   [gamma=0 collapses to inhibitor form]
dratio_top4_bottom4 = inhib_conc_matrix_top4_bottom4 / ic50_matrix_top4_bottom4
y_true_top4_bottom4 = np.where(
    valid_matrix_top4_bottom4,
    (1 + gamma_matrix_top4_bottom4 * dratio_top4_bottom4) / (1 + dratio_top4_bottom4),
    1.0,
)

print(f"y_true (activity g) shape: {y_true_top4_bottom4.shape}")
print(pd.DataFrame(y_true_top4_bottom4, index=modules, columns=exp_list).round(3).to_string())


y_true (activity g) shape: (7, 12)
        QS-11  navitoclax  BMS-536924  AT9283  TG-101348  CYT-387  ruxolitinib  Deforolimus  Temsirolimus  Sapanisertib  Pevonedistat  Thapsigargin
ARFGAP   0.13       1.000       1.000   1.000      1.000    1.000          1.0        1.000           1.0           1.0         1.000         1.000
BCL      1.00       0.002       1.000   1.000      1.000    1.000          1.0        1.000           1.0           1.0         1.000         1.000
IGF1R    1.00       1.000       0.079   1.000      1.000    1.000          1.0        1.000           1.0           1.0         1.000         1.000
JAK      1.00       1.000       1.000   0.038      0.041    0.099          0.0        1.000           1.0           1.0         1.000         1.000
MTOR     1.00       1.000       1.000   1.000      1.000    1.000          1.0        0.002           0.0           0.0         1.000         1.000
NAE      1.00       1.000       1.000   1.000      1.000    1.000          1.

In [ ]:
pert_matrix_top4_bottom4 = np.zeros((len(modules), len(exp_list)))
for _, row in drug_gene.iterrows():
    if row["module"] in modules and row["drug"] in exp_list:
        i = modules.index(row["module"])
        j = exp_list.index(row["drug"])
        pert_matrix_top4_bottom4[i, j] = 1

fit_mask_top4_bottom4 = pert_matrix_top4_bottom4 * valid_matrix_top4_bottom4
print(f"pert_matrix shape: {pert_matrix_top4_bottom4.shape}")
print(pd.DataFrame(pert_matrix_top4_bottom4.astype(int), index=modules, columns=exp_list).to_string())
print(f"\nfit_mask (excludes missing-IC50 pairs):")
print(pd.DataFrame(fit_mask_top4_bottom4.astype(int), index=modules, columns=exp_list).to_string())


pert_matrix shape: (7, 12)
        QS-11  navitoclax  BMS-536924  AT9283  TG-101348  CYT-387  ruxolitinib  Deforolimus  Temsirolimus  Sapanisertib  Pevonedistat  Thapsigargin
ARFGAP      1           0           0       0          0        0            0            0             0             0             0             0
BCL         0           1           0       0          0        0            0            0             0             0             0             0
IGF1R       0           0           1       0          0        0            0            0             0             0             0             0
JAK         0           0           0       1          1        1            1            0             0             0             0             0
MTOR        0           0           0       0          0        0            0            1             1             1             0             0
NAE         0           0           0       0          0        0            0       

In [ ]:
residuals, a_coeffs = cstarpy.integration.pathway_activity.prediction.predict_coeffs(
    x, y_true_top4_bottom4, fit_mask_top4_bottom4,
    200_000, 10, 10, 10, 100
)

a_coeffs_df_top4_bottom4 = pd.DataFrame(a_coeffs, index=modules, columns=genes)
a_coeffs_df_top4_bottom4.to_csv(os.path.join(out_dir, "a_coeffs_df_top4_bottom4.csv"))
print(f"a_coeffs shape: {a_coeffs.shape}")
trh = 0.0001


100%|██████████| 200000/200000 [05:38<00:00, 591.09it/s]


a_coeffs shape: (7, 15045)

Genes representing each module:
ARFGAP     1
BCL        2
IGF1R      6
JAK       35
MTOR       2
NAE        4
SERCA      4


In [ ]:
# Use .values for matrix math (a_coeffs as array, not DataFrame)
a_coeffs_top4_bottom4 = a_coeffs_df_top4_bottom4.values

pathway_activity_top4_bottom4 = a_coeffs_top4_bottom4 @ x
pd.DataFrame(pathway_activity_top4_bottom4, index=modules, columns=exp_list)\
    .to_csv(os.path.join(out_dir, "pathway_activity_top4_bottom4.csv"))

R_global_top4_bottom4 = cstarpy.integration.pathway_activity.calc_global_response_from_pathway_activity(
    cstarpy.integration.pathway_activity.calc_pathway_activity(x, a_coeffs_top4_bottom4),
    modules, exp_list
)
R_global_df_top4_bottom4 = pd.DataFrame(R_global_top4_bottom4, index=modules, columns=exp_list)
R_global_df_top4_bottom4.to_csv(os.path.join(out_dir, "R_global_core_top4_bottom4.csv"))

pd.DataFrame(y_true_top4_bottom4,      index=modules, columns=exp_list).to_csv(os.path.join(out_dir, "y_true_top4_bottom4.csv"))
pd.DataFrame(pert_matrix_top4_bottom4, index=modules, columns=exp_list).to_csv(os.path.join(out_dir, "pert_matrix_top4_bottom4.csv"))
x_df.to_csv(os.path.join(out_dir, "Data_norm.csv"))


Global response matrix:
        QS-11  navitoclax  BMS-536924  AT9283  TG-101348  CYT-387  ruxolitinib  Deforolimus  Temsirolimus  Sapanisertib  Pevonedistat  Thapsigargin
ARFGAP -1.252       0.000      -0.004   0.000     -0.001   -0.001       -0.003        0.184        -0.003        -0.005        -0.002        -0.333
BCL     0.000      -1.595       0.000   0.000     -0.000    0.001       -0.000       -0.000        -0.000        -0.001        -0.000        -0.755
IGF1R  -0.002       0.000      -1.442  -0.000     -0.003   -0.004       -0.010       -0.001        -0.003        -0.004        -0.001        -0.003
JAK    -1.210       0.002      -1.899  -1.146     -1.530   -1.346       -1.902       -0.012        -1.175        -1.897        -0.637        -1.987
MTOR   -0.001      -0.000      -0.003  -0.000     -0.001   -0.002       -0.013       -1.627        -1.919        -1.972        -0.000        -0.007
NAE    -0.750      -0.000      -0.001  -0.000     -0.001   -0.000       -0.002       -0.